# 🧪 Lab 11: Monitoring & Performance

**Topics:** explain(), $indexStats, collStats, serverStatus, profiler

---

In [ ]:
from pymongo import MongoClient, ReadPreference
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern
from pymongo.errors import ConnectionFailure
from bson import ObjectId
from datetime import datetime, timedelta, timezone
import pprint

USE_ATLAS = False
ATLAS_URI  = "mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority"
DOCKER_URI = "mongodb://127.0.0.1:27017/?directConnection=true"

uri = ATLAS_URI if USE_ATLAS else DOCKER_URI
client = MongoClient(uri, serverSelectionTimeoutMS=5000)
try:
    client.admin.command("ping")
    print("✅ Connected to", "Atlas" if USE_ATLAS else "Docker MongoDB")
except ConnectionFailure as e:
    print("❌ Connection failed:", e)

db = client["mongo_labs"]
pp = pprint.PrettyPrinter(indent=2)

## explain() — Query execution

In [ ]:
plan = db.orders.find({"status":"delivered"}).explain("executionStats")
print("Stage:", plan["queryPlanner"]["winningPlan"]["stage"])
print("Docs examined:", plan["executionStats"]["totalDocsExamined"])
print("Docs returned:", plan["executionStats"]["totalDocsReturned"])
print("Time (ms):", plan["executionStats"]["executionTimeMillis"])

## $indexStats — Index usage

In [ ]:
print("Index usage stats:")
for stat in db.orders.aggregate([{"$indexStats":{}}]):
    print(f"  {stat['name']:30s} accesses={stat['accesses']['ops']}"))

## collStats — Collection size

In [ ]:
stats = db.command("collStats","orders")
print("=== orders collection ===" )
print(f"  Count: {stats.get('count',0)}")
print(f"  Storage: {stats.get('storageSize',0)} bytes")
print(f"  Index size: {stats.get('totalIndexSize',0)} bytes")
print(f"  Indexes: {len(stats.get('indexSizes',{}))}")

## dbStats — Database stats

In [ ]:
db_stats = db.command("dbStats")
print("=== mongo_labs database ===" )
print(f"  Collections: {db_stats.get('collections',0)}")
print(f"  Objects: {db_stats.get('objects',0)}")
print(f"  Data size: {db_stats.get('dataSize',0)} bytes")

## serverStatus — Server metrics

In [ ]:
admin_db = client["admin"]
svr = admin_db.command("serverStatus")
print("=== Server Status ===" )
print(f"  Connections current: {svr['connections']['current']}")
print(f"  Connections available: {svr['connections']['available']}")
print(f"  Insert ops: {svr['opcounters']['insert']}")
print(f"  Query ops: {svr['opcounters']['query']}")

## currentOp — Running operations

In [ ]:
ops = admin_db.command("currentOp",{"active":True,"secs_running":{"$gt":1}})
if ops["inprog"]:
    for op in ops["inprog"]:
        print(f"  opid={op.get('opid')}, secs={op.get('secs_running')}")
else:
    print("✅ No long-running operations")

## Profiler — Capture slow queries

In [ ]:
db.command("profile",1,slowms=100)
db.orders.find({"status":"delivered"}).sort("total",-1).toArray()
profiled = list(db.system.profile.find({}).sort("ts",-1).limit(5))
if profiled:
    for op in profiled[:3]:
        print(f"  {op.get('op')}: {op.get('millis')}ms examined={op.get('docsExamined')}")
db.command("profile",0)
print("✅ Profiler off")